# Lens Notebook

This notebook shows how to use both `LogitLens` and `TunedLens` with `ModelWithSplitPoints` for language modeling and sequence classification.

Origins and related work:
- Logit Lens: [nostalgebraist (2020), *interpreting GPT: the logit lens*](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)
- Tuned Lens: [Belrose et al. (2023), *Eliciting Latent Predictions from Transformers with the Tuned Lens*](https://arxiv.org/abs/2303.08112)
- Related vocabulary-space analysis: [Geva et al. (2022), *Transformer Feed-Forward Layers Build Predictions by Promoting Concepts in the Vocabulary Space*](https://aclanthology.org/2022.emnlp-main.3/)

The tuned lens implementation supports three initialization modes:
- `logit_lens`: identity initialization so tuning starts from the plain logit-lens behavior
- `xavier`: Xavier uniform initialization with zero bias
- `default`: the current `torch.nn.Linear` initialization

The notebook uses tiny checkpoints so it stays light enough for quick experimentation.
Some of them are random checkpoints, so semantic quality is not the goal here: the examples are mainly meant to illustrate the API and the decodability metrics. The examples split at complete transformer-block outputs, where the hidden states belong to the residual stream expected by the model's output head.

Metric interpretation:
- `mean_target_score`: higher is better
- `target_cross_entropy`: lower is better
- `perplexity`: lower is better for causal language models
- `kl_divergence_to_model`: lower is better and differentiable, which makes it useful as a regularization target for linear decodability
- `model_top1_agreement`: agreement with the final model argmax

Intermediate softmax values are reported as scores, not calibrated probabilities. Hidden-state distributions can differ substantially from the final layer distribution on which the output head was trained.

The raw `explain()` output is tensor-first and uses `top_indices` / `top_scores`.
Human-readable decoding is handled separately by `plot_lens()`.


In [1]:
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    set_seed,
)

from interpreto import LogitLens, ModelWithSplitPoints, TunedLens, plot_lens

set_seed(0)


def summarize_metrics(metrics, split_point):
    keys = [
        "target_source",
        "nb_evaluated_elements",
        "mean_target_score",
        "target_cross_entropy",
        "target_accuracy",
        "mean_max_score",
        "kl_divergence_to_model",
        "model_top1_agreement",
        "perplexity",
    ]
    return {key: metrics[split_point][key] for key in keys if key in metrics[split_point]}

## Causal Language Model

We start with a small GPT-style model and inspect two prompts at once.


In [2]:
causal_model_name = "hf-internal-testing/tiny-random-gpt2"
causal_model = AutoModelForCausalLM.from_pretrained(causal_model_name)
causal_tokenizer = AutoTokenizer.from_pretrained(causal_model_name)
if causal_tokenizer.pad_token is None:
    causal_tokenizer.pad_token = causal_tokenizer.eos_token

causal_model_with_split_points = ModelWithSplitPoints(
    causal_model,
    tokenizer=causal_tokenizer,
    split_point="transformer.h.1",
    batch_size=2,
    device_map="cpu",
)

causal_examples = [
    "Interpreto is useful.",
    "Interpreto helps explain models.",
]

causal_model_with_split_points.split_point

'transformer.h.1'

In [3]:
causal_logit_lens = LogitLens(causal_model_with_split_points, top_k=3)
causal_model_inputs = causal_tokenizer(causal_examples, return_tensors="pt", padding=True, truncation=True)
causal_logit_explanations = causal_logit_lens.explain(causal_model_inputs)
plot_lens(
    causal_logit_explanations,
    causal_model_inputs,
    tokenizer=causal_tokenizer,
    task=causal_logit_lens.task,
)

In [4]:
(
    causal_logit_explanations["transformer.h.1"]["top_indices"][0, 0],
    causal_logit_explanations["transformer.h.1"]["top_scores"][0, 0],
)

(tensor([841, 708, 914]), tensor([0.0014, 0.0014, 0.0013]))

In [5]:
causal_logit_metrics = causal_logit_lens.metrics(causal_model_inputs)
summarize_metrics(causal_logit_metrics, "transformer.h.1")

{'target_source': 'next_token',
 'nb_evaluated_elements': 28,
 'mean_target_score': 0.0009728240547701716,
 'target_cross_entropy': 6.940088748931885,
 'target_accuracy': 0.0,
 'mean_max_score': 0.0014791741268709302,
 'kl_divergence_to_model': 0.002218021312728524,
 'model_top1_agreement': 0.2857142984867096,
 'perplexity': 1032.861876281221}

## Masked Language Model

The same `LogitLens` workflow also works on masked-language-model checkpoints.


In [6]:
masked_model_name = "hf-internal-testing/tiny-random-bert"
masked_config = AutoConfig.from_pretrained(masked_model_name)
masked_model = AutoModelForMaskedLM.from_config(masked_config)
masked_tokenizer = AutoTokenizer.from_pretrained(masked_model_name)

masked_model_with_split_points = ModelWithSplitPoints(
    masked_model,
    tokenizer=masked_tokenizer,
    split_point="bert.encoder.layer.1.output",
    batch_size=2,
    device_map="cpu",
)

masked_examples = [
    "Interpreto is useful",
    "Interpreto explains transformers",
]

masked_model_with_split_points.split_point

'bert.encoder.layer.1.output'

In [7]:
masked_logit_lens = LogitLens(masked_model_with_split_points, top_k=4)
masked_model_inputs = masked_tokenizer(masked_examples, return_tensors="pt", padding=True, truncation=True)
masked_logit_explanations = masked_logit_lens.explain(masked_model_inputs)
plot_lens(
    masked_logit_explanations,
    masked_model_inputs,
    tokenizer=masked_tokenizer,
    task=masked_logit_lens.task,
)

In [8]:
masked_logit_metrics = masked_logit_lens.metrics(masked_model_inputs)
summarize_metrics(masked_logit_metrics, "bert.encoder.layer.1.output")

{'target_source': 'token_identity',
 'nb_evaluated_elements': 48,
 'mean_target_score': 0.0008612762321718037,
 'target_cross_entropy': 7.060431003570557,
 'target_accuracy': 0.0,
 'mean_max_score': 0.0013121230294927955,
 'kl_divergence_to_model': 1.8744127601166838e-06,
 'model_top1_agreement': 0.8958333134651184}

## Sequence Classification

For classification, the same framework exposes intermediate label distributions and classification-oriented scores.
To keep this notebook lightweight and warning-free, the example below builds a small two-label classifier from the tiny BERT configuration instead of loading mismatched task heads from a generic checkpoint.


In [9]:
classification_config = AutoConfig.from_pretrained(masked_model_name)
classification_config.num_labels = 2
classification_config.id2label = {0: "negative", 1: "positive"}
classification_config.label2id = {"negative": 0, "positive": 1}
classification_model = AutoModelForSequenceClassification.from_config(classification_config)
classification_tokenizer = AutoTokenizer.from_pretrained(masked_model_name)

classification_model_with_split_points = ModelWithSplitPoints(
    classification_model,
    tokenizer=classification_tokenizer,
    split_point="bert.encoder.layer.1.output",
    batch_size=2,
    device_map="cpu",
)

classification_examples = [
    "Interpreto is helpful",
    "Interpreto is practical",
]
classification_label_names = {0: "negative", 1: "positive"}
classification_targets = [1, 0]

classification_model_with_split_points.split_point

'bert.encoder.layer.1.output'

In [10]:
classification_logit_lens = LogitLens(classification_model_with_split_points, top_k=2)
classification_model_inputs = classification_tokenizer(
    classification_examples, return_tensors="pt", padding=True, truncation=True
)
classification_logit_explanations = classification_logit_lens.explain(classification_model_inputs)
plot_lens(
    classification_logit_explanations,
    classification_model_inputs,
    tokenizer=classification_tokenizer,
    task=classification_logit_lens.task,
    label_names=classification_label_names,
)

In [11]:
classification_logit_metrics = classification_logit_lens.metrics(
    classification_model_inputs,
    targets=classification_targets,
)
summarize_metrics(classification_logit_metrics, "bert.encoder.layer.1.output")

{'target_source': 'provided_targets',
 'nb_evaluated_elements': 2,
 'mean_target_score': 0.499994695186615,
 'target_cross_entropy': 0.6931830644607544,
 'target_accuracy': 0.5,
 'mean_max_score': 0.5035551190376282,
 'kl_divergence_to_model': 0.0,
 'model_top1_agreement': 1.0}

## Tuned Lens On A Small Dataset

The final section fits a `TunedLens` on a tiny text collection for the causal model.
This is only a small demonstration, but it shows how the decodability metrics can be tracked before and after tuning. The comparison and final plot use two held-out sentences that are not part of the fitting collection.


In [12]:
supported_modes = ["logit_lens", "xavier", "default"]
[
    TunedLens(causal_model_with_split_points, top_k=3, initialization_mode=mode).initialization_mode
    for mode in supported_modes
]

['logit_lens', 'xavier', 'default']

In [13]:
tuning_texts = [
    "Interpreto is useful.",
    "Interpreto helps explain transformers.",
    "Interpreto makes debugging easier.",
    "Interpreto is practical for analysis.",
]

held_out_texts = [
    "Interpreto helps debug transformers.",
    "Interpreto makes analysis practical.",
]
held_out_model_inputs = causal_tokenizer(held_out_texts, return_tensors="pt", padding=True, truncation=True)

tuned_lens = TunedLens(causal_model_with_split_points, top_k=3, initialization_mode="logit_lens")
pre_tuning_metrics = summarize_metrics(tuned_lens.metrics(held_out_model_inputs), "transformer.h.1")
history = tuned_lens.fit(tuning_texts, epochs=2, batch_size=2)
post_tuning_metrics = summarize_metrics(tuned_lens.metrics(held_out_model_inputs), "transformer.h.1")
history

{'loss': [0.002205893157550426, 0.001996013269857017],
 'split_point': 'transformer.h.1',
 'epochs': 2}

In [14]:
{"before": pre_tuning_metrics, "after": post_tuning_metrics}

{'before': {'target_source': 'next_token',
  'nb_evaluated_elements': 35,
  'mean_target_score': 0.0009856362594291568,
  'target_cross_entropy': 6.9274468421936035,
  'target_accuracy': 0.0,
  'mean_max_score': 0.0014687504153698683,
  'kl_divergence_to_model': 0.002215180778875947,
  'model_top1_agreement': 0.2571428716182709,
  'perplexity': 1019.8867209243274},
 'after': {'target_source': 'next_token',
  'nb_evaluated_elements': 35,
  'mean_target_score': 0.0009856006363406777,
  'target_cross_entropy': 6.926970481872559,
  'target_accuracy': 0.0,
  'mean_max_score': 0.0014693881385028362,
  'kl_divergence_to_model': 0.0018508885987102985,
  'model_top1_agreement': 0.34285715222358704,
  'perplexity': 1019.4010030560631}}

In [15]:
tuned_lens_explanations = tuned_lens.explain(held_out_model_inputs)
plot_lens(
    tuned_lens_explanations,
    held_out_model_inputs,
    tokenizer=causal_tokenizer,
    task=tuned_lens.task,
)

In [16]:
held_out_metrics = tuned_lens.metrics(held_out_model_inputs)
summarize_metrics(held_out_metrics, "transformer.h.1")

{'target_source': 'next_token',
 'nb_evaluated_elements': 35,
 'mean_target_score': 0.0009856006363406777,
 'target_cross_entropy': 6.926970481872559,
 'target_accuracy': 0.0,
 'mean_max_score': 0.0014693881385028362,
 'kl_divergence_to_model': 0.0018508885987102985,
 'model_top1_agreement': 0.34285715222358704,
 'perplexity': 1019.4010030560631}